# NYC Food Safety Intelligence Dashboard

## Milestone 3 Final Notebook

This notebook turns the earlier milestone work into a more polished, presentation-ready dashboard analysis for New York City restaurant food safety.

### Milestone 3 goals covered here
- strengthen the data-cleaning pipeline
- use meaningful colors that communicate safe vs risky outcomes clearly
- create decision-friendly visualizations for class presentation
- highlight actionable insights for diners, students, and tourists
- export clean summary tables for a future localhost dashboard

### Color logic used throughout
- **Green** = safer / stronger outcome
- **Amber** = caution / medium concern
- **Red** = worse / higher risk
- **Blue** = neutral trend / informational context


## Chunk 1: Import Libraries and Define a Consistent Visual Language

### Why I am doing this
For Milestone 3, I want the dashboard to feel intentional and easy to interpret during presentation. Before doing any analysis, I define the main libraries and a color system that stays consistent across all charts.

### Why this matters for the project
This project is about helping users interpret restaurant safety patterns quickly. If colors change meaning from one chart to another, the dashboard becomes confusing. By defining the palette up front, I make the visuals more trustworthy and easier to explain.


In [6]:
import os
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

SAFE_GREEN = "#2E8B57"
CAUTION_AMBER = "#E3A008"
ALERT_RED = "#D64545"
INFO_BLUE = "#2563EB"
SLATE = "#475569"
LIGHT_BG = "#F8FAFC"

GRADE_COLORS = {
    "A": SAFE_GREEN,
    "B": CAUTION_AMBER,
    "C": ALERT_RED,
}

RISK_COLORS = {
    "Low": SAFE_GREEN,
    "Medium": CAUTION_AMBER,
    "High": ALERT_RED,
}

PLOT_TEMPLATE = "plotly_white"

print("Libraries and presentation color palette loaded.")


ModuleNotFoundError: No module named 'ipywidgets'

## Chunk 2: Load the Dataset with a Flexible File Path

### Why I am doing this
I want the notebook to work both on my own computer and inside the project workspace. Instead of relying on only one hard-coded path, I check a few possible dataset locations.

### Why this matters for the project
This makes the notebook more reproducible. A professor or teammate can understand exactly where the data is expected to come from, and the notebook is less likely to fail because of a path issue.


In [ ]:
DATA_PATH_OPTIONS = [
    Path("/Users/shahbazshaikh/Desktop/indata_sp26/nyc_food_safety:/nyc_inspection_data.csv"),
    Path("nyc_inspection_data.csv"),
    Path("data/nyc_inspection_data.csv"),
]

DATA_PATH = next((path for path in DATA_PATH_OPTIONS if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find nyc_inspection_data.csv. Update DATA_PATH_OPTIONS in this notebook."
    )

raw = pd.read_csv(DATA_PATH)
print(f"Loaded dataset from: {DATA_PATH}")
print("Raw shape:", raw.shape)
raw.head()


## Chunk 3: Clean and Standardize the Inspection Data

### Why I am doing this
The raw NYC inspection file includes inconsistent column names, repeated inspections for the same restaurant, missing values, and non-standard formatting in fields like score, ZIP code, and dates. This chunk standardizes the dataset so the later visuals are based on comparable values.

### Why this matters for the project
Without cleaning, the dashboard could produce misleading results. For example, scores would not be comparable if some records were missing dates or if grades outside A, B, and C were mixed into the same charts. This chunk creates a reliable foundation for every later milestone insight.


In [ ]:
df = raw.copy()
df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

rename_map = {
    "boro": "borough",
    "zip": "zipcode",
    "bldg": "building",
}
df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})

if "inspection_date" in df.columns:
    df["inspection_date"] = pd.to_datetime(df["inspection_date"], errors="coerce")
if "grade_date" in df.columns:
    df["grade_date"] = pd.to_datetime(df["grade_date"], errors="coerce")
if "record_date" in df.columns:
    df["record_date"] = pd.to_datetime(df["record_date"], errors="coerce")
if "score" in df.columns:
    df["score"] = pd.to_numeric(df["score"], errors="coerce")
if "zipcode" in df.columns:
    df["zipcode"] = (
        df["zipcode"].astype(str).str.extract(r"(\d{5})", expand=False)
    )

text_cols = [
    "dba", "borough", "cuisine_description", "inspection_type",
    "action", "grade", "critical_flag", "violation_description"
]
for col in text_cols:
    if col in df.columns:
        df[col] = (
            df[col].astype(str)
            .str.strip()
            .replace({"": np.nan, "nan": np.nan, "NaN": np.nan})
        )

usable = df.dropna(subset=["inspection_date", "inspection_type"]).copy()
usable = usable[usable["grade"].isin(["A", "B", "C"])].copy()
usable = usable.drop_duplicates()

usable["borough"] = usable["borough"].fillna("Unknown")
usable["cuisine_description"] = usable["cuisine_description"].fillna("Unknown")
usable["dba"] = usable["dba"].fillna("Unknown Restaurant")
usable["year"] = usable["inspection_date"].dt.year
usable["month"] = usable["inspection_date"].dt.month
usable["year_month"] = usable["inspection_date"].dt.to_period("M").astype(str)

def risk_bucket(score):
    if pd.isna(score):
        return np.nan
    if score <= 13:
        return "Low"
    if score <= 27:
        return "Medium"
    return "High"

usable["risk_level"] = usable["score"].apply(risk_bucket)
usable["has_critical_violation"] = usable["critical_flag"].str.lower().eq("critical")

latest = (
    usable.sort_values(["camis", "inspection_date"], ascending=[True, False])
    .drop_duplicates(subset=["camis"], keep="first")
    .copy()
)

latest["latitude"] = pd.to_numeric(latest["latitude"], errors="coerce")
latest["longitude"] = pd.to_numeric(latest["longitude"], errors="coerce")

map_df = latest[
    latest["latitude"].between(40.45, 40.95)
    & latest["longitude"].between(-74.30, -73.65)
].copy()

print("Cleaned analysis shape:", usable.shape)
print("Latest restaurant snapshot:", latest.shape)
print("Map-ready restaurants:", map_df.shape)
usable.head()


## Data Quality Notes

- The raw file contains repeated inspections for the same restaurant, so the notebook keeps both:
  - a full cleaned inspection table for trends
  - a **latest inspection snapshot** for restaurant-level comparisons
- Grades were limited to **A, B, and C** so the class presentation focuses on interpretable public-facing outcomes.
- Scores are treated as **lower is better**, which drives the green-to-red encoding in summary charts.


## Chunk 4: Build High-Level KPI Metrics for the Dashboard

### Why I am doing this
Before moving into detailed charts, I summarize the overall state of restaurant safety in a few headline metrics. These include how many restaurants are in the latest snapshot, what share earned an A, the average inspection score, and how often critical violations appear.

### Why this matters for the project
This chunk gives the audience a fast overview of the citywide situation. It works well as an opening slide or dashboard header because it frames the rest of the analysis in a concise and presentation-friendly way.


In [ ]:
kpi_restaurants = latest["camis"].nunique()
kpi_grade_a_pct = (latest["grade"].eq("A").mean() * 100)
kpi_avg_score = latest["score"].mean()
kpi_critical_pct = (latest["has_critical_violation"].mean() * 100)

fig_kpi = go.Figure()
fig_kpi.add_trace(go.Indicator(
    mode="number",
    value=kpi_restaurants,
    title={"text": "Restaurants in Latest Snapshot"},
    number={"font": {"color": SLATE, "size": 36}},
    domain={"row": 0, "column": 0},
))
fig_kpi.add_trace(go.Indicator(
    mode="number",
    value=kpi_grade_a_pct,
    title={"text": "A Grade Share (%)"},
    number={"suffix": "%", "font": {"color": SAFE_GREEN, "size": 36}},
    domain={"row": 0, "column": 1},
))
fig_kpi.add_trace(go.Indicator(
    mode="number",
    value=kpi_avg_score,
    title={"text": "Average Inspection Score"},
    number={"font": {"color": INFO_BLUE, "size": 36}, "valueformat": ".1f"},
    domain={"row": 0, "column": 2},
))
fig_kpi.add_trace(go.Indicator(
    mode="number",
    value=kpi_critical_pct,
    title={"text": "Critical Violation Share (%)"},
    number={"suffix": "%", "font": {"color": ALERT_RED, "size": 36}, "valueformat": ".1f"},
    domain={"row": 0, "column": 3},
))
fig_kpi.update_layout(
    grid={"rows": 1, "columns": 4, "pattern": "independent"},
    template=PLOT_TEMPLATE,
    height=260,
    margin=dict(l=20, r=20, t=50, b=20),
    title="Milestone 3 Dashboard Summary"
)
fig_kpi.show()


## Chunk 5: Compare Boroughs Using Average Inspection Score

### Why I am doing this
One of the most natural questions in a citywide food safety project is whether safety outcomes vary across boroughs. I group restaurants by borough and compare their latest average inspection scores.

### Why this matters for the project
This chunk supports geographic comparison, which is important for the dashboard’s driving question. I also use a green-to-red scale here because lower scores are better, so the color directly reinforces the meaning of the values.


In [ ]:
borough_summary = (
    latest.groupby("borough", as_index=False)
    .agg(
        avg_score=("score", "mean"),
        restaurant_count=("camis", "nunique"),
        critical_rate=("has_critical_violation", "mean"),
    )
)
borough_summary["critical_rate"] = borough_summary["critical_rate"] * 100
borough_summary = borough_summary.sort_values("avg_score", ascending=True)

fig_borough = px.bar(
    borough_summary,
    x="borough",
    y="avg_score",
    color="avg_score",
    color_continuous_scale="RdYlGn_r",
    text="restaurant_count",
    template=PLOT_TEMPLATE,
    title="Average Inspection Score by Borough"
)
fig_borough.update_traces(
    texttemplate="n=%{text}",
    textposition="outside",
    hovertemplate=(
        "<b>%{x}</b><br>"
        "Average score: %{y:.2f}<br>"
        "Restaurants: %{text}<extra></extra>"
    )
)
fig_borough.update_layout(
    xaxis_title="Borough",
    yaxis_title="Average Score (Lower = Better)",
    coloraxis_colorbar_title="Avg Score",
    height=520
)
fig_borough.show()

borough_summary


## Chunk 6: Show the Grade Mix by Borough

### Why I am doing this
Average score alone does not tell the full story. To make the borough comparison easier for a broader audience, I also show the grade distribution within each borough.

### Why this matters for the project
Grades are more familiar to most viewers than raw inspection scores. Using green for A, amber for B, and red for C creates an intuitive summary that is especially useful in a live class presentation.


In [ ]:
grade_mix = (
    latest.groupby(["borough", "grade"])
    .size()
    .reset_index(name="count")
)
grade_totals = grade_mix.groupby("borough")["count"].transform("sum")
grade_mix["share_pct"] = grade_mix["count"] / grade_totals * 100

fig_grade_mix = px.bar(
    grade_mix,
    x="borough",
    y="share_pct",
    color="grade",
    barmode="stack",
    color_discrete_map=GRADE_COLORS,
    template=PLOT_TEMPLATE,
    title="Grade Mix by Borough"
)
fig_grade_mix.update_layout(
    xaxis_title="Borough",
    yaxis_title="Share of Restaurants (%)",
    legend_title="Grade",
    height=520
)
fig_grade_mix.update_traces(
    hovertemplate=(
        "<b>%{x}</b><br>"
        "Grade %{legendgroup}: %{y:.1f}%<extra></extra>"
    )
)
fig_grade_mix.show()


## Chunk 7: Track How Inspection Scores Change Over Time

### Why I am doing this
Milestone 3 should go beyond static comparisons and show temporal patterns. This chunk groups inspections by month to examine whether average scores improve, worsen, or remain stable over time.

### Why this matters for the project
A time trend adds depth to the dashboard because it shows that restaurant safety is not just a location-based issue. It also helps answer whether recent conditions look different from earlier periods in the dataset.


In [ ]:
monthly_trend = (
    usable.dropna(subset=["score"])
    .groupby("year_month", as_index=False)
    .agg(
        avg_score=("score", "mean"),
        inspections=("camis", "count")
    )
)
monthly_trend = monthly_trend[monthly_trend["year_month"] >= "2018-01"].copy()

fig_trend = px.line(
    monthly_trend,
    x="year_month",
    y="avg_score",
    markers=True,
    template=PLOT_TEMPLATE,
    title="Average Inspection Score Over Time"
)
fig_trend.update_traces(
    line=dict(color=INFO_BLUE, width=3),
    marker=dict(size=6, color=INFO_BLUE),
    hovertemplate=(
        "<b>%{x}</b><br>"
        "Average score: %{y:.2f}<extra></extra>"
    )
)
fig_trend.update_layout(
    xaxis_title="Year-Month",
    yaxis_title="Average Score (Lower = Better)",
    height=520
)
fig_trend.show()


## Chunk 8: Identify Cuisines with Higher Critical Violation Rates

### Why I am doing this
Cuisine categories are an important business and public-health dimension in this dataset. In this chunk, I compare cuisines by critical violation rate, while requiring a minimum number of restaurants so that the comparison is more stable.

### Why this matters for the project
This chunk moves the dashboard from simple description toward pattern discovery. It helps show whether some cuisine groups appear more frequently in higher-risk inspection outcomes, while avoiding over-interpreting very small categories.


In [ ]:
cuisine_summary = (
    latest.groupby("cuisine_description", as_index=False)
    .agg(
        restaurants=("camis", "nunique"),
        avg_score=("score", "mean"),
        critical_rate=("has_critical_violation", "mean"),
    )
)
cuisine_summary["critical_rate"] = cuisine_summary["critical_rate"] * 100
cuisine_summary = cuisine_summary[cuisine_summary["restaurants"] >= 30].copy()

top_cuisines = cuisine_summary.sort_values("critical_rate", ascending=False).head(15)

fig_cuisine = px.bar(
    top_cuisines.sort_values("critical_rate", ascending=True),
    x="critical_rate",
    y="cuisine_description",
    orientation="h",
    color="critical_rate",
    color_continuous_scale=[[0, SAFE_GREEN], [0.5, CAUTION_AMBER], [1, ALERT_RED]],
    text="restaurants",
    template=PLOT_TEMPLATE,
    title="Top 15 Cuisines by Critical Violation Rate"
)
fig_cuisine.update_traces(
    texttemplate="n=%{text}",
    textposition="outside",
    hovertemplate=(
        "<b>%{y}</b><br>"
        "Critical violation rate: %{x:.1f}%<br>"
        "Restaurants: %{text}<extra></extra>"
    )
)
fig_cuisine.update_layout(
    xaxis_title="Critical Violation Rate (%)",
    yaxis_title="Cuisine",
    coloraxis_colorbar_title="Critical %",
    height=650
)
fig_cuisine.show()


## Chunk 9: Build an Interactive Restaurant Safety Map with Clean Hover Labels and Filters

### Why I am doing this
An interactive map makes the project feel more like a real dashboard rather than a static report. I use the latest restaurant snapshot with valid latitude and longitude values so users can explore food safety outcomes spatially.

### Why this matters for the project
This is one of the strongest presentation visuals because it combines location, score, grade, cuisine, and risk level in one place. I also add dropdown filters for borough and restaurant type so the dashboard is easier to explore during presentation, and I replace the default hover formatting with cleaner labels that read more like a finished product.


In [ ]:
map_df["restaurant_name"] = map_df["dba"].fillna("Unknown Restaurant")
map_df["cuisine_clean"] = map_df["cuisine_description"].fillna("Unknown").str.title()
map_df["inspection_day"] = map_df["inspection_date"].dt.strftime("%b %d, %Y")
map_df["borough_clean"] = map_df["borough"].fillna("Unknown")

borough_options = ["All Boroughs"] + sorted(map_df["borough_clean"].dropna().unique().tolist())
cuisine_options = ["All Restaurant Types"] + sorted(map_df["cuisine_clean"].dropna().unique().tolist())

borough_dropdown = widgets.Dropdown(
    options=borough_options,
    value="All Boroughs",
    description="Borough:",
    layout=widgets.Layout(width="320px")
)

cuisine_dropdown = widgets.Dropdown(
    options=cuisine_options,
    value="All Restaurant Types",
    description="Type:",
    layout=widgets.Layout(width="360px")
)

output_map = widgets.Output()

def render_map(selected_borough, selected_cuisine):
    filtered = map_df.copy()

    if selected_borough != "All Boroughs":
        filtered = filtered[filtered["borough_clean"] == selected_borough]

    if selected_cuisine != "All Restaurant Types":
        filtered = filtered[filtered["cuisine_clean"] == selected_cuisine]

    with output_map:
        output_map.clear_output(wait=True)

        if filtered.empty:
            print("No restaurants match the selected filters.")
            return

        fig_map = px.scatter_map(
            filtered,
            lat="latitude",
            lon="longitude",
            color="risk_level",
            color_discrete_map=RISK_COLORS,
            custom_data=[
                "restaurant_name",
                "borough_clean",
                "cuisine_clean",
                "grade",
                "score",
                "inspection_day",
                "risk_level",
            ],
            zoom=10,
            height=720,
            title="NYC Restaurant Safety Map (Latest Inspection per Restaurant)"
        )

        fig_map.update_traces(
            marker={"size": 9, "opacity": 0.80},
            hovertemplate=(
                "<b>%{customdata[0]}</b><br>"
                "Borough: %{customdata[1]}<br>"
                "Restaurant Type: %{customdata[2]}<br>"
                "Grade: %{customdata[3]}<br>"
                "Score: %{customdata[4]:.0f}<br>"
                "Inspection Date: %{customdata[5]}<br>"
                "Risk Level: %{customdata[6]}"
                "<extra></extra>"
            ),
        )

        fig_map.update_layout(
            template=PLOT_TEMPLATE,
            legend_title_text="Risk Level",
            margin=dict(l=10, r=10, t=60, b=10)
        )
        fig_map.show()

def update_from_widgets(change=None):
    render_map(borough_dropdown.value, cuisine_dropdown.value)

borough_dropdown.observe(update_from_widgets, names="value")
cuisine_dropdown.observe(update_from_widgets, names="value")

display(widgets.HBox([borough_dropdown, cuisine_dropdown]))
display(output_map)
render_map("All Boroughs", "All Restaurant Types")


## Chunk 10: Create a “Quick Safer Picks” Table for Real Users

### Why I am doing this
The dashboard should not only describe the data, but also translate it into something practical. This chunk identifies restaurants with an A grade, low risk level, and no critical violation in their latest inspection.

### Why this matters for the project
This chunk shows the applied value of the project. Instead of only discussing patterns, it demonstrates how a diner, tourist, or student could actually use the dashboard to make a safer dining decision.


In [ ]:
quick_picks = latest[
    (latest["grade"] == "A")
    & (latest["risk_level"] == "Low")
    & (~latest["has_critical_violation"])
].copy()

quick_picks = quick_picks.sort_values(
    ["score", "inspection_date"],
    ascending=[True, False]
)

quick_picks_view = quick_picks[
    ["dba", "borough", "cuisine_description", "grade", "score", "inspection_date"]
].head(20).copy()
quick_picks_view["inspection_date"] = quick_picks_view["inspection_date"].dt.strftime("%Y-%m-%d")

fig_table = go.Figure(
    data=[
        go.Table(
            header=dict(
                values=[
                    "Restaurant", "Borough", "Cuisine", "Grade", "Score", "Inspection Date"
                ],
                fill_color=SAFE_GREEN,
                font=dict(color="white", size=13),
                align="left",
            ),
            cells=dict(
                values=[
                    quick_picks_view["dba"],
                    quick_picks_view["borough"],
                    quick_picks_view["cuisine_description"],
                    quick_picks_view["grade"],
                    quick_picks_view["score"],
                    quick_picks_view["inspection_date"],
                ],
                fill_color=LIGHT_BG,
                align="left",
                height=28,
            ),
        )
    ]
)
fig_table.update_layout(
    title="Quick Safer Picks for Diners",
    height=650,
    margin=dict(l=10, r=10, t=50, b=10),
)
fig_table.show()


## Chunk 11: Summarize the Most Common Violation Descriptions

### Why I am doing this
To understand what drives inspection problems, I also look directly at the violation text descriptions that appear most often in the cleaned inspection data.

### Why this matters for the project
This chunk helps connect high-level dashboard metrics back to concrete food safety issues. It gives the professor and the audience a clearer sense of what kinds of problems inspectors are repeatedly finding.


In [ ]:
top_violations = (
    usable["violation_description"]
    .dropna()
    .value_counts()
    .head(12)
    .reset_index()
)
top_violations.columns = ["violation_description", "count"]

fig_viol = px.bar(
    top_violations.sort_values("count", ascending=True),
    x="count",
    y="violation_description",
    orientation="h",
    color="count",
    color_continuous_scale=[[0, CAUTION_AMBER], [1, ALERT_RED]],
    template=PLOT_TEMPLATE,
    title="Most Common Violation Descriptions"
)
fig_viol.update_layout(
    xaxis_title="Number of Inspection Records",
    yaxis_title="Violation Description",
    coloraxis_colorbar_title="Count",
    height=680
)
fig_viol.show()


## Milestone 3 Interpretation

### Key takeaways
- Borough score differences are visible, but all boroughs still include both strong and weak restaurant outcomes.
- Grade distributions are easier to explain when color is meaningful: **A = green**, **B = amber**, **C = red**.
- Cuisine comparisons become more honest when filtered to cuisines with enough restaurants to make the comparison stable.
- The map is most useful for presentation because it connects restaurant location, grade, score, and risk in one view.

### Class presentation angle
- Start with the KPI row to summarize the citywide picture.
- Move to boroughs and grades to compare neighborhoods.
- Use the cuisine and violation charts to explain what types of problems appear most often.
- End with the map and safer-picks table to show how the analysis helps real diners make decisions.


## Chunk 12: Export Milestone 3 Tables for Future Dashboard Use

### Why I am doing this
The final step is to save cleaned summary tables that can be reused in a localhost web dashboard or future app version of the project.

### Why this matters for the project
This chunk makes the notebook more than a one-time analysis. It creates structured outputs that can support the next stage of the project, including a browser-based dashboard for presentation.


In [ ]:
output_dir = Path("outputs_milestone3")
output_dir.mkdir(exist_ok=True)

borough_summary.to_csv(output_dir / "borough_summary.csv", index=False)
monthly_trend.to_csv(output_dir / "monthly_trend.csv", index=False)
top_cuisines.to_csv(output_dir / "cuisine_critical_rate.csv", index=False)
quick_picks_view.to_csv(output_dir / "quick_safer_picks.csv", index=False)
map_df[[
    "camis", "restaurant_name", "borough", "cuisine_clean", "grade",
    "score", "risk_level", "inspection_date", "latitude", "longitude"
]].to_csv(output_dir / "map_snapshot.csv", index=False)

print(f"Milestone 3 outputs saved to: {output_dir.resolve()}")
